In [1]:
import pandas as pd
import os

In [2]:
pwd

'/home/trungdt2/Documents/GIS_VTB_Project2025/crawl_all_khoang_cach'

In [3]:
ls

 data/                        run_schedule_A.sh*
 data_20260313/               run_schedule_B.sh*
 etl_20250604/                split_to_part_100.py
 HUONG_DAN_SU_DUNG_CRAWL.md  'Tạo_tổ_hợp_mới 2026-Copy1.ipynb'
 LICH_CHAY_SONG_SONG.md      'Tạo_tổ_hợp_mới 2026.ipynb'
 part/                        tinh_mat_do_file_cu.ipynb
 part_2025/                   TONG_QUAN.md
 run_crawl_part_B.py          top_1_gan_nhat_trong_tinh.ipynb
 run_crawl_part.ipynb         top_1_tinh_cu_moi.ipynb
 run_crawl_part.py            xu_ly_data_kc_20250520.ipynb


In [4]:
ml_2025 = pd.read_excel(r'./etl_20250604/full_trong_tinh_sau_sat_nhap_20250520.xlsx')

In [5]:
ml_2025.head()

,Mã phòng ban 1,Tên phòng ban 1,KINH ĐỘ 1,VĨ ĐỘ 1,Mã phòng ban 2,Tên phòng ban 2,KINH ĐỘ 2,VĨ ĐỘ 2,Khoảng cách đường bộ (km),Khoảng cách chim bay (km),Tỉnh thành (sau sáp nhập)
0,36039000,PGD Bồ Xuyên,106.336744,20.449196,36041000,PGD Hoàng Công Chất,106.331829,20.438444,1.6,1.300629,Hưng Yên
1,36039000,PGD Bồ Xuyên,106.336744,20.449196,36042000,PGD Chợ Đậu,106.343223,20.433061,2.0,1.916853,Hưng Yên
2,36039000,PGD Bồ Xuyên,106.336744,20.449196,36044000,PGD Vũ Thư,106.288684,20.435725,5.8,5.226741,Hưng Yên
3,36040000,PGD Quỳnh Phụ,106.327291,20.660420,36041000,PGD Hoàng Công Chất,106.331829,20.438444,28.8,24.687114,Hưng Yên
4,36040000,PGD Quỳnh Phụ,106.327291,20.660420,36044000,PGD Vũ Thư,106.288684,20.435725,31.8,25.306250,Hưng Yên


In [6]:
import unicodedata
import re

def normalize_column_name(col):
    # chuyển Đ đ trước khi remove accent
    col = col.replace("Đ", "D").replace("đ", "d")
    
    # bỏ dấu tiếng Việt
    col = unicodedata.normalize("NFKD", col)
    col = "".join(c for c in col if not unicodedata.combining(c))
    
    # lower
    col = col.lower()
    
    # bỏ khoảng trắng đầu cuối
    col = col.strip()
    
    # thay khoảng trắng bằng _
    col = re.sub(r"\s+", "_", col)
    
    # thay . bằng _
    col = col.replace(".", "_")
    
    # bỏ ký tự đặc biệt
    col = re.sub(r"[^a-z0-9_]", "", col)
    
    # bỏ _ trùng nhau
    col = re.sub(r"_+", "_", col)
    
    return col

In [7]:
# chuẩn hóa tên cột của df bỏ dấu lower
ml_2025.columns = [normalize_column_name(col) for col in ml_2025.columns]

In [8]:
ml_2025.head()

,ma_phong_ban_1,ten_phong_ban_1,kinh_do_1,vi_do_1,ma_phong_ban_2,ten_phong_ban_2,kinh_do_2,vi_do_2,khoang_cach_duong_bo_km,khoang_cach_chim_bay_km,tinh_thanh_sau_sap_nhap
0,36039000,PGD Bồ Xuyên,106.336744,20.449196,36041000,PGD Hoàng Công Chất,106.331829,20.438444,1.6,1.300629,Hưng Yên
1,36039000,PGD Bồ Xuyên,106.336744,20.449196,36042000,PGD Chợ Đậu,106.343223,20.433061,2.0,1.916853,Hưng Yên
2,36039000,PGD Bồ Xuyên,106.336744,20.449196,36044000,PGD Vũ Thư,106.288684,20.435725,5.8,5.226741,Hưng Yên
3,36040000,PGD Quỳnh Phụ,106.327291,20.660420,36041000,PGD Hoàng Công Chất,106.331829,20.438444,28.8,24.687114,Hưng Yên
4,36040000,PGD Quỳnh Phụ,106.327291,20.660420,36044000,PGD Vũ Thư,106.288684,20.435725,31.8,25.306250,Hưng Yên


In [9]:
df_final = ml_2025.copy()
df_final['ma_don_vi_1'] = ml_2025['ma_phong_ban_1']//100000
df_final['ma_don_vi_2'] = ml_2025['ma_phong_ban_2']//100000

In [14]:
import pandas as pd

# =========================
# 1. Copy data
# =========================
df = df_final.copy()

# =========================
# 2. Bỏ self-pair nếu có
# =========================
df = df[df['ma_phong_ban_1'] != df['ma_phong_ban_2']].copy()

# =========================
# 3. Nếu có cột mã đơn vị thì tạo cờ cùng đơn vị
# =========================
if 'ma_don_vi_1' in df.columns and 'ma_don_vi_2' in df.columns:
    df['cung_don_vi'] = df['ma_don_vi_1'] == df['ma_don_vi_2']
else:
    df['cung_don_vi'] = False

# =========================
# 4. Hàm gộp tên
# =========================
def gop_ten(series):
    vals = (
        series.dropna()
        .astype(str)
        .str.strip()
    )
    vals = vals[vals != '']
    vals = pd.unique(vals)
    return '; '.join(vals)

# =========================
# 5. Hàm tính số lượng unique ma_phong_ban_2
# =========================
def tinh_so_luong(df_in, col_khoang_cach, nguong, ten_cot):
    return (
        df_in[df_in[col_khoang_cach] <= nguong]
        .groupby('ma_phong_ban_1')['ma_phong_ban_2']
        .nunique()
        .rename(ten_cot)
    )

# =========================
# 6. Hàm tính số lượng cùng đơn vị
# =========================
def tinh_so_luong_cung_dv(df_in, col_khoang_cach, nguong, ten_cot):
    return (
        df_in[(df_in[col_khoang_cach] <= nguong) & (df_in['cung_don_vi'])]
        .groupby('ma_phong_ban_1')['ma_phong_ban_2']
        .nunique()
        .rename(ten_cot)
    )

# =========================
# 7. Hàm gộp tên phòng ban
# =========================
def tinh_danh_sach_ten(df_in, col_khoang_cach, nguong, ten_cot):
    return (
        df_in[df_in[col_khoang_cach] <= nguong]
        .groupby('ma_phong_ban_1')['ten_phong_ban_2']
        .apply(gop_ten)
        .rename(ten_cot)
    )

# =========================
# 8. Hàm gộp tên phòng ban cùng đơn vị
# =========================
def tinh_danh_sach_ten_cung_dv(df_in, col_khoang_cach, nguong, ten_cot):
    return (
        df_in[(df_in[col_khoang_cach] <= nguong) & (df_in['cung_don_vi'])]
        .groupby('ma_phong_ban_1')['ten_phong_ban_2']
        .apply(gop_ten)
        .rename(ten_cot)
    )

# ==========================================================
# 9. TÍNH THEO ĐƯỜNG BỘ: dùng cột khoang_cach_duong_bo_km
# ==========================================================
sl_db_05 = tinh_so_luong(df, 'khoang_cach_duong_bo_km', 0.5, 'SL_diem_duong_bo_0.5km')
sl_db_1  = tinh_so_luong(df, 'khoang_cach_duong_bo_km', 1.0, 'SL_diem_duong_bo_1km')

pb_db_05 = tinh_danh_sach_ten(df, 'khoang_cach_duong_bo_km', 0.5, 'Phongban_duong_bo_0.5km')
pb_db_1  = tinh_danh_sach_ten(df, 'khoang_cach_duong_bo_km', 1.0, 'Phongban_duong_bo_1km')

sl_db_05_dv = tinh_so_luong_cung_dv(df, 'khoang_cach_duong_bo_km', 0.5, 'SL_diem_duong_bo_0.5km_cung_donvi')
sl_db_1_dv  = tinh_so_luong_cung_dv(df, 'khoang_cach_duong_bo_km', 1.0, 'SL_diem_duong_bo_1km_cung_donvi')

pb_db_05_dv = tinh_danh_sach_ten_cung_dv(df, 'khoang_cach_duong_bo_km', 0.5, 'Phongban_duong_bo_0.5km_cung_donvi')
pb_db_1_dv  = tinh_danh_sach_ten_cung_dv(df, 'khoang_cach_duong_bo_km', 1.0, 'Phongban_duong_bo_1km_cung_donvi')

# ==========================================================
# 10. TÍNH THEO ĐƯỜNG CHIM BAY: dùng cột khoang_cach_chim_bay_km
# ==========================================================
sl_cb_05 = tinh_so_luong(df, 'khoang_cach_chim_bay_km', 0.5, 'SL_diem_chim_bay_0.5km')
sl_cb_1  = tinh_so_luong(df, 'khoang_cach_chim_bay_km', 1.0, 'SL_diem_chim_bay_1km')

pb_cb_05 = tinh_danh_sach_ten(df, 'khoang_cach_chim_bay_km', 0.5, 'Phongban_chim_bay_0.5km')
pb_cb_1  = tinh_danh_sach_ten(df, 'khoang_cach_chim_bay_km', 1.0, 'Phongban_chim_bay_1km')

sl_cb_05_dv = tinh_so_luong_cung_dv(df, 'khoang_cach_chim_bay_km', 0.5, 'SL_diem_chim_bay_0.5km_cung_donvi')
sl_cb_1_dv  = tinh_so_luong_cung_dv(df, 'khoang_cach_chim_bay_km', 1.0, 'SL_diem_chim_bay_1km_cung_donvi')

pb_cb_05_dv = tinh_danh_sach_ten_cung_dv(df, 'khoang_cach_chim_bay_km', 0.5, 'Phongban_chim_bay_0.5km_cung_donvi')
pb_cb_1_dv  = tinh_danh_sach_ten_cung_dv(df, 'khoang_cach_chim_bay_km', 1.0, 'Phongban_chim_bay_1km_cung_donvi')

# =========================
# 11. Tạo bảng gốc 1 dòng / 1 ma_phong_ban_1
# =========================
df_ket_qua = (
    df[
        [
            'ma_phong_ban_1',
            'ten_phong_ban_1',
            'kinh_do_1',
            'vi_do_1',
            'tinh_thanh_sau_sap_nhap'
        ]
    ]
    .drop_duplicates()
    .copy()
)

# =========================
# 12. Merge toàn bộ kết quả
# =========================
list_series = [
    sl_db_05, sl_db_1, pb_db_05, pb_db_1,
    sl_db_05_dv, sl_db_1_dv, pb_db_05_dv, pb_db_1_dv,
    sl_cb_05, sl_cb_1, pb_cb_05, pb_cb_1,
    sl_cb_05_dv, sl_cb_1_dv, pb_cb_05_dv, pb_cb_1_dv
]

for s in list_series:
    df_ket_qua = df_ket_qua.merge(s, on='ma_phong_ban_1', how='left')

# =========================
# 13. Fillna
# =========================
cols_num = [
    'SL_diem_duong_bo_0.5km',
    'SL_diem_duong_bo_1km',
    'SL_diem_duong_bo_0.5km_cung_donvi',
    'SL_diem_duong_bo_1km_cung_donvi',
    'SL_diem_chim_bay_0.5km',
    'SL_diem_chim_bay_1km',
    'SL_diem_chim_bay_0.5km_cung_donvi',
    'SL_diem_chim_bay_1km_cung_donvi'
]

cols_text = [
    'Phongban_duong_bo_0.5km',
    'Phongban_duong_bo_1km',
    'Phongban_duong_bo_0.5km_cung_donvi',
    'Phongban_duong_bo_1km_cung_donvi',
    'Phongban_chim_bay_0.5km',
    'Phongban_chim_bay_1km',
    'Phongban_chim_bay_0.5km_cung_donvi',
    'Phongban_chim_bay_1km_cung_donvi'
]

for c in cols_num:
    if c in df_ket_qua.columns:
        df_ket_qua[c] = df_ket_qua[c].fillna(0).astype(int)

for c in cols_text:
    if c in df_ket_qua.columns:
        df_ket_qua[c] = df_ket_qua[c].fillna('')

# =========================
# 14. Kết quả
# =========================
print(df_ket_qua.shape)
df_ket_qua.head()

(1068, 21)


,ma_phong_ban_1,ten_phong_ban_1,kinh_do_1,vi_do_1,tinh_thanh_sau_sap_nhap,SL_diem_duong_bo_0.5km,SL_diem_duong_bo_1km,Phongban_duong_bo_0.5km,Phongban_duong_bo_1km,SL_diem_duong_bo_0.5km_cung_donvi,...,Phongban_duong_bo_0.5km_cung_donvi,Phongban_duong_bo_1km_cung_donvi,SL_diem_chim_bay_0.5km,SL_diem_chim_bay_1km,Phongban_chim_bay_0.5km,Phongban_chim_bay_1km,SL_diem_chim_bay_0.5km_cung_donvi,SL_diem_chim_bay_1km_cung_donvi,Phongban_chim_bay_0.5km_cung_donvi,Phongban_chim_bay_1km_cung_donvi
0,36039000,PGD Bồ Xuyên,106.336744,20.449196,Hưng Yên,1,2,PGD Minh Khai,CN Thái Bình; PGD Minh Khai,1,...,PGD Minh Khai,CN Thái Bình; PGD Minh Khai,1,3,PGD Minh Khai,CN Thái Bình; PGD Minh Khai; PGD Trần Hưng Đạo,1,3,PGD Minh Khai,CN Thái Bình; PGD Minh Khai; PGD Trần Hưng Đạo
1,36040000,PGD Quỳnh Phụ,106.327291,20.660420,Hưng Yên,0,0,,,0,...,,,0,0,,,0,0,,
2,36041000,PGD Hoàng Công Chất,106.331829,20.438444,Hưng Yên,0,1,,PGD Trần Hưng Đạo,0,...,,PGD Trần Hưng Đạo,1,1,PGD Trần Hưng Đạo,PGD Trần Hưng Đạo,1,1,PGD Trần Hưng Đạo,PGD Trần Hưng Đạo
3,36042000,PGD Chợ Đậu,106.343223,20.433061,Hưng Yên,0,0,,,0,...,,,0,0,,,0,0,,
4,38000000,CN Nam Định,106.175814,20.431630,Ninh Bình,0,1,,PGD Nguyễn Du,0,...,,,1,2,PGD Nguyễn Du,CN Bắc Nam Định; PGD Nguyễn Du,0,0,,


In [15]:
df_ket_qua.to_excel(r'/home/trungdt2/Documents/GIS_VTB_Project2025/crawl_all_khoang_cach/etl_20250604/template_sheet_1_2025.xlsx',index=False)